# Kaggle 性別預測模型建立完整流程
根據 `data_processing_pipeline_phases_0_5.md` 實作可執行的 Pipeline。

In [ ]:
# !pip install imbalanced-learn missingpy xgboost catboost optuna -q

In [1]:
# import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import classification_report, f1_score, roc_auc_score
from imblearn.under_sampling import EditedNearestNeighbours
from imblearn.combine import SMOTEENN
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

## 第一步：準備資料與預防資料洩漏
讀取資料並先進行 Train/Test Split (抽出驗證集)。

In [2]:
train_path = 'data/boy or girl 2025 train_missingValue.csv'
test_path = 'data/boy or girl 2025 test no ans_missingValue.csv'

df_train_full = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

X_full = df_train_full.drop(columns=['gender'])
y_full = df_train_full['gender']

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42, stratify=y_full
)
print('Train shape:', X_train.shape)
print('Valid shape:', X_valid.shape)
print('Test shape:', df_test.shape)

Train shape: (338, 10)
Valid shape: (85, 10)
Test shape: (426, 11)


## Phase 0, 1, 2: 資料清理、異常值處理與特徵萃取

In [ ]:
def is_repeated_or_symbol(s):
    if pd.isna(s) or s == '' or s == 'nan': return False
    if re.fullmatch(r'[^A-Za-z0-9\u4e00-\u9fff]+', s): return True
    if re.fullmatch(r'(.)\1{3,}', s): return True
    return False

def build_features(df):
    df = df.copy()
    
    # Phase 0: yt & phone_os
    if 'yt' in df.columns:
        orig_yt = df['yt']
        df['yt'] = pd.to_numeric(df['yt'], errors='coerce')
        df['is_yt_invalid'] = np.where(orig_yt.notna() & df['yt'].isna(), 1, 0)
    
    if 'phone_os' in df.columns:
        df['phone_os_clean'] = df['phone_os'].astype(str).str.strip().str.lower().replace({'iphone': 'ios'})
        valid_os = {'android', 'ios', 'windows'}
        df['is_phone_os_invalid'] = np.where(df['phone_os'].notna() & ~df['phone_os_clean'].isin(valid_os), 1, 0)
        df.loc[df['is_phone_os_invalid'] == 1, 'phone_os_clean'] = 'Unknown'
        df.drop(columns=['phone_os'], inplace=True, errors='ignore')

    # Phase 1: 異常值標記與截斷
    if 'weight' in df.columns:
        df['is_weight_missing'] = df['weight'].isna().astype(int)
        df['is_weight_outlier'] = np.where(df['weight'].notna() & ((df['weight'] < 30) | (df['weight'] > 1000)), 1, 0)
        df['weight'] = df['weight'].clip(lower=40, upper=120)

    if 'height' in df.columns:
        df['is_height_missing'] = df['height'].isna().astype(int)
        df['is_height_outlier'] = np.where(df['height'].notna() & ((df['height'] < 130) | (df['height'] > 220)), 1, 0)
        df['height'] = df['height'].clip(lower=140, upper=200)

    if 'iq' in df.columns:
        df['is_iq_outlier'] = np.where(df['iq'].notna() & ((df['iq'] < 100) | (df['iq'] > 140)), 1, 0)
        df['iq'] = df['iq'].clip(lower=100, upper=140)

    if 'fb_friends' in df.columns:
        df['is_fb_friends_outlier'] = np.where(df['fb_friends'].notna() & ((df['fb_friends'] < 0) | (df['fb_friends'] > 2000)), 1, 0)
        df['fb_friends'] = df['fb_friends'].clip(lower=0, upper=2000)

    if 'yt' in df.columns:
        df['is_yt_outlier'] = np.where(df['yt'].notna() & ((df['yt'] < 0) | (df['yt'] > 24)), 1, df.get('is_yt_invalid', 0))
        df['yt'] = df['yt'].clip(lower=0, upper=24)

    if 'star_sign' in df.columns:
        valid_stars = {'aries', 'taurus', 'gemini', 'cancer', 'leo', 'virgo', 'libra', 'scorpio', 'sagittarius', 'capricorn', 'aquarius', 'pisces'}
        df['star_sign_clean'] = df['star_sign'].astype(str).str.strip().str.lower()
        df['is_star_sign_invalid'] = np.where(df['star_sign'].notna() & ~df['star_sign_clean'].isin(valid_stars), 1, 0)
        df.loc[df['is_star_sign_invalid'] == 1, 'star_sign_clean'] = 'Unknown'
        df.drop(columns=['star_sign'], inplace=True, errors='ignore')

    # Phase 2: 自我介紹文本特徵
    if 'self_intro' in df.columns:
        df['is_intro_missing'] = df['self_intro'].isna().astype(int)
        df['intro_char_length'] = df['self_intro'].fillna('').astype(str).str.len()
        df['intro_word_count'] = df['self_intro'].fillna('').astype(str).str.split().apply(len)
        df['intro_is_all_lower'] = df['self_intro'].fillna('').astype(str).apply(lambda s: s.islower() if s else False).astype(int)
        df['intro_is_all_upper'] = df['self_intro'].fillna('').astype(str).apply(lambda s: s.isupper() if s else False).astype(int)

        df['is_intro_anomalous'] = 0
        df.loc[df['intro_char_length'] == 0, 'is_intro_anomalous'] = 1
        df.loc[df['intro_char_length'] > 500, 'is_intro_anomalous'] = 1
        df.loc[df['self_intro'].fillna('').astype(str).apply(is_repeated_or_symbol), 'is_intro_anomalous'] = 1
        
        # 將異常文字設為 NaN 後續給 Imputation，或者直接捨棄該欄
        df.loc[df['is_intro_anomalous'] == 1, 'self_intro'] = np.nan
        df.drop(columns=['self_intro'], inplace=True, errors='ignore')
        
    return df

X_train_clean = build_features(X_train)
X_valid_clean = build_features(X_valid)
X_test_clean = build_features(df_test)

print("特徵工程完成")

特徵工程完成


In [4]:
X_train_clean.head()

,id,height,weight,sleepiness,iq,fb_friends,yt,is_yt_invalid,phone_os_clean,is_phone_os_invalid,...,is_fb_friends_outlier,is_yt_outlier,star_sign_clean,is_star_sign_invalid,is_intro_missing,intro_char_length,intro_word_count,intro_is_all_lower,intro_is_all_upper,is_intro_anomalous
145,146,168.0,63.0,2.0,100.0,328.0,2.0,0,android,0,...,0,0,Unknown,1,1,0,0,0,0,1
78,79,170.0,NaN,NaN,100.0,99.0,10.0,0,android,0,...,0,0,Unknown,1,0,4,1,1,0,0
16,17,156.0,47.0,NaN,130.0,400.0,3.5,0,Unknown,1,...,0,0,Unknown,1,0,26,5,0,0,0
149,150,168.0,49.0,1.0,100.0,2000.0,24.0,0,nan,0,...,0,1,Unknown,1,0,2,1,0,0,0
293,294,NaN,76.0,4.0,140.0,1287.0,24.0,0,Unknown,1,...,0,1,nan,0,0,17,4,0,0,0


## Phase 3: MissForest 缺失值填補與類別特徵處理
利用 IterativeImputer (基於 Random Forest，此為 sklearn 原生穩定版本，表現等價於 missingpy 的 MissForest) 對數值補值。

In [ ]:
# 分離數值與類別特徵 (需先確認欄位存在)
num_cols = [c for c in ['height', 'weight', 'iq', 'fb_friends', 'yt', 'sleepiness'] if c in X_train_clean.columns]
cat_cols = [c for c in ['phone_os_clean', 'star_sign_clean'] if c in X_train_clean.columns]

# 類別變數填補 Mode 或 'Unknown'
for c in cat_cols:
    mode = X_train_clean[c].mode()[0] if len(X_train_clean[c].mode()) > 0 else 'Unknown'
    X_train_clean[c].fillna(mode, inplace=True)
    X_valid_clean[c].fillna(mode, inplace=True)
    X_test_clean[c].fillna(mode, inplace=True)

# 數值使用 MissForest (IterativeImputer with RandomForest)
# 備註：為避免 missingpy 與最新版本 sklearn 相容性問題，這裡採用等價的 sklearn API 開發：
rf_estimator = RandomForestRegressor(n_estimators=50, random_state=42)
miss_forest = IterativeImputer(estimator=rf_estimator, random_state=42, max_iter=10)

if len(num_cols) > 0:
    X_train_clean[num_cols] = miss_forest.fit_transform(X_train_clean[num_cols])
    X_valid_clean[num_cols] = miss_forest.transform(X_valid_clean[num_cols])
    X_test_clean[num_cols]  = miss_forest.transform(X_test_clean[num_cols])

# 對類別變數進行 One-Hot Encode
X_train_enc = pd.get_dummies(X_train_clean, columns=cat_cols, dummy_na=True)
X_valid_enc = pd.get_dummies(X_valid_clean, columns=cat_cols, dummy_na=True)
X_test_enc  = pd.get_dummies(X_test_clean, columns=cat_cols, dummy_na=True)

# 對齊欄位 (訓練集與測試集 One-hot 欄位同步)
X_train_enc, X_valid_enc = X_train_enc.align(X_valid_enc, join='left', axis=1, fill_value=0)
X_train_enc, X_test_enc  = X_train_enc.align(X_test_enc, join='left', axis=1, fill_value=0)

# 保留並移除 id 欄位 (不再納入訓練)
train_ids = X_train_enc.pop('id') if 'id' in X_train_enc.columns else None
valid_ids = X_valid_enc.pop('id') if 'id' in X_valid_enc.columns else None
test_ids  = X_test_enc.pop('id') if 'id' in X_test_enc.columns else None

print("缺失值填補與特徵編碼完成，特徵數:", X_train_enc.shape[1])

In [ ]:
missing_train = X_train_enc.isnull().sum()
missing_valid = X_valid_enc.isnull().sum()
missing_test = X_test_enc.isnull().sum()

print("訓練集剩餘缺失值 (大於 0 的欄位)：")
print(missing_train[missing_train > 0])
print("\n驗證集剩餘缺失值 (大於 0 的欄位)：")
print(missing_valid[missing_valid > 0])
print("\n測試集剩餘缺失值 (大於 0 的欄位)：")
print(missing_test[missing_test > 0])


## Phase 4: 資料重採樣 (ENN)
使用 ENN 清理男女分類邊界上的雜訊樣本

In [ ]:
print("重採樣前樣本數:", X_train_enc.shape[0])
print("原比例:\n", y_train.value_counts())

# 使用 ENN，針對邊界雜訊進行清除 (保留決策邊界清晰的資料)
enn = EditedNearestNeighbours(n_neighbors=3)
X_train_resampled, y_train_resampled = enn.fit_resample(X_train_enc, y_train)

print("重採樣後樣本數:", X_train_resampled.shape[0])
print("新比例:\n", y_train_resampled.value_counts())

## Phase 5: 特徵選擇與模型訓練 (Embedded Method)
先用 Random Forest 評估特徵重要性，留下最具預測力的核心特徵，隨後進行最終模型訓練。

In [ ]:
fs_model = RandomForestClassifier(n_estimators=100, random_state=42)
fs_model.fit(X_train_resampled, y_train_resampled)

feat_imp = pd.Series(fs_model.feature_importances_, index=X_train_resampled.columns).sort_values(ascending=False)

plt.figure(figsize=(10,6))
feat_imp.head(20).plot(kind='bar')
plt.title("Top 20 Feature Importances")
plt.show()

# 篩選重要性 > 0 的核心特徵
selected_features = feat_imp[feat_imp > 0].index.tolist()
print(f"從 {X_train_resampled.shape[1]} 個特徵中選出 {len(selected_features)} 個核心特徵。")

X_train_sel = X_train_resampled[selected_features]
X_valid_sel = X_valid_enc[selected_features]
X_test_sel  = X_test_enc[selected_features]

### 最終模型訓練與評估 (XGBoost)
用最精煉的特徵與樣本，訓練最終的預測模型 (利用 XGBoost 建立非線性決策邊界)。

In [ ]:
from sklearn.model_selection import GridSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from xgboost import XGBClassifier

# === 【進階】包含 MissForest 的全管線 GridSearchCV ===
# 為了同時尋找 MissForest 與 XGBoost 的最佳參數，
# 我們必須把它們放入 Pipeline 中，並且傳入**尚未補值的資料**。
# 這裡我們使用先前尚未做 MissForest 補值，但已經過初步清洗 (build_features) 且分離出數值與類別的原始 DataFrame：X_train_clean 來示範。

print("設定 Pipeline：包含 MissForest(IterativeImputer) 與 XGBoost")

# 定義基底 Imputer 與 模型
rf_estimator = RandomForestRegressor(random_state=42)
miss_forest = IterativeImputer(estimator=rf_estimator, random_state=42)

base_xgb = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

# 建立管道 (這裡為了示範速度，暫時省略 ENN，只對 Imputer 和 XGB 做調參)
pipe = ImbPipeline([
    ('imputer', miss_forest),
    ('xgb', base_xgb)
])

# XGBoost 需要連續標籤 (0, 1)
labels_offset_cv = y_train.min()
y_train_cv = y_train - labels_offset_cv
y_valid_cv = y_valid - labels_offset_cv

# 準備可輸入 Pipeline 的資料。
# 注意：為了避免維度不一致，這裡我們直接拿 One-Hot Encode 後，但裡面【仍有數值 NaN 的副本】來進行 CV。
# 建立帶有 NaN 的 X_train_nan (將連續欄位標為 NaN)
X_train_nan = X_train_enc.copy()
X_valid_nan = X_valid_enc.copy()
# (在前面的儲存格中 X_train_enc 的原始數值已經被補值過了，為了示範這裡的 GridSearch 邏輯：
# 你應該在呼叫 IterativeImputer 前保留一份未補值的 X_train_nan，並拿來餵給 GridSearchCV)

# 由於前面已經把 X_train_enc 整個補滿了，若要真正調參 MissForest，必須回到未補值前的狀態：
# 取出 build_features 後未補值的連續變數與已經補 mode 的類別變數
X_train_nan_features = X_train_clean.copy()
X_valid_nan_features = X_valid_clean.copy()

# 把類別欄位做 One-Hot (保留數值欄位的 NaN)
X_train_cv_enc = pd.get_dummies(X_train_nan_features, columns=cat_cols, dummy_na=True)
X_valid_cv_enc = pd.get_dummies(X_valid_nan_features, columns=cat_cols, dummy_na=True)
X_train_cv_enc, X_valid_cv_enc = X_train_cv_enc.align(X_valid_cv_enc, join='left', axis=1, fill_value=0)

if 'id' in X_train_cv_enc.columns: X_train_cv_enc.drop(columns=['id'], inplace=True)
if 'id' in X_valid_cv_enc.columns: X_valid_cv_enc.drop(columns=['id'], inplace=True)

# 設定參數網格 (Grid Search)
# 注意：同時跑 Imputer 調參會非常耗時，這裡給予較精簡的參數做示範。
param_grid = {
    # ==========================================
    # 1. MissForest 填補器參數 (著重於提升填補穩定度)
    # ==========================================
    'imputer__max_iter': [5, 10],    
    'imputer__estimator__n_estimators': [30, 50], 
    'imputer__estimator__max_depth': [3, 5, None], # 關鍵：加入淺層樹 (3, 5)，避免內部 RF 死背異常特徵
    'imputer__estimator__min_samples_leaf': [1, 2, 4], # 新增：控制葉節點最小樣本數，防止填補出極端雜訊
    
    # ==========================================
    # 2. XGBoost 分類器參數 (小資料集特化防禦機制)
    # ==========================================
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__n_estimators': [100, 200, 300],
    
    # 【控制模型複雜度】
    'xgb__max_depth': [2, 3, 4], # 修正：砍掉太深的樹。對於 400 筆資料，深度 5 已經極度危險，2~4 是最佳甜區
    
    # 【引入隨機性 (Stochastic Gradient Boosting)】
    'xgb__subsample': [0.6, 0.8, 1.0],       # 樣本採樣率：每次建樹隨機抽取 60%~80% 的資料，強迫模型學習不同面向
    'xgb__colsample_bytree': [0.6, 0.8, 1.0], # 特徵採樣率：每次建樹隨機抽取特徵，避免模型過度依賴單一強勢特徵 (如體重)
    
    # 【正則化懲罰項 (Regularization)】
    'xgb__reg_alpha': [0, 0.1, 1],  # L1 正則化 (Lasso)：具有特徵選擇效果，會把不重要的特徵權重直接壓到 0
    'xgb__reg_lambda': [1, 5, 10]   # L2 正則化 (Ridge)：平滑權重，對抗小資料集中極端值 (亂填數據) 的干擾
}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring='f1_macro',   
    cv=3,                 
    verbose=2,
    n_jobs=-1   # 若電腦支援可開多核心加速
)

print("開始執行全管線(MissForest + XGBoost) GridSearchCV...")
# 為了避免示範執行過久，建議只在選定特徵上執行。這裡直接對所有欄位訓練。
grid_search.fit(X_train_cv_enc, y_train_cv)

print("\nBest Parameters found by GridSearchCV:")
print(grid_search.best_params_)
print("Best cross-validation F1-Score (Macro): {:.4f}".format(grid_search.best_score_))

# 取得最好的模型 Pipeline
final_pipeline = grid_search.best_estimator_

# 在驗證集上評估
y_pred = final_pipeline.predict(X_valid_cv_enc) + labels_offset_cv

print("\nValidation F1-Score (Macro):", f1_score(y_valid, y_pred, average='macro'))
print("\nClassification Report:\n", classification_report(y_valid, y_pred))

# == 如果要產出 submission == 
# final_model_for_sub = final_pipeline


## 產生 Kaggle Submission 檔案

In [ ]:
if test_ids is not None:
    # 準備包含 NaN 但已做 One-Hot 的 Test Data (與 CV 一致)
    X_test_nan_features = X_test_clean.copy()
    X_test_cv_enc = pd.get_dummies(X_test_nan_features, columns=cat_cols, dummy_na=True)
    # 與訓練集對齊特徵
    _, X_test_cv_enc = X_train_cv_enc.align(X_test_cv_enc, join='left', axis=1, fill_value=0)
    
    if 'id' in X_test_cv_enc.columns: X_test_cv_enc.drop(columns=['id'], inplace=True)
    
    # 這裡的 final_pipeline 包含了 tuned 的 imputer 和 XGBoost
    test_preds = final_pipeline.predict(X_test_cv_enc) + labels_offset_cv

    submission = pd.DataFrame({
        'id': test_ids,
        'gender': test_preds
    })

    submission.to_csv('submission.csv', index=False)
    print("✅ submission.csv 儲存成功！資料夾預覽：")
    print(submission.head())
else:
    print("Error: Test data id column not found...")